# Quantization

Store a float tensor as small integers plus a float scale, and reconstruct on use:

$$w \approx s \cdot q, \qquad q \in [-Q, Q] \subset \text{int8}, \qquad Q = 127$$

Symmetric (no zero point), so the scale is set by the largest magnitude in the group:

$$s = \frac{\max |w|}{Q}, \qquad q = \text{round}\!\left(\frac{w}{s}\right)$$

Rounding to the nearest of $2Q+1$ levels bounds the reconstruction error at half a step:

$$|w - s \cdot q| \le \frac{s}{2}$$

The **group** is the whole choice. `dim=-1` scales per row of the weight matrix
(per-channel); `dim=None` scales the whole matrix at once (per-tensor). Under one
shared scale a quiet row gets only $Q \cdot \frac{\max|w_{\text{row}}|}{\max|w|}$
levels — the rest of the range is reserved for a magnitude it never reaches.

Implementation and its fast tests: `src/video/quantize.py`. This notebook holds the
sweeps too slow to run on every test.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.append(str(root / "src"))
sys.path.append(str(root / "src" / "video"))

import math

import torch

from checkpoint import load_checkpoint
from dataset import BinDataset
from evaluate import bits_per_char, full_loss
from paths import CKPT_DIR
from quantize import (
    dequantize,
    nbytes,
    quantizable,
    quantize_model,
    quantize_packed,
    unpack_bits,
)

CKPT = CKPT_DIR / "big_2026-08-30_09-09-16.pt"
DEV = "cuda" if torch.cuda.is_available() else "cpu"


def load():
    """A fresh fp32 copy of the trained 27M checkpoint, in eval mode."""
    m, _ = load_checkpoint(CKPT, DEV)
    return m.eval()


def kl_vs(ref_logprobs, model, x):
    """KL(fp32 || quantized) over the same batch, in nats per token."""
    with torch.no_grad():
        lp = model(x).float().log_softmax(-1)
    return (ref_logprobs.exp() * (ref_logprobs - lp)).sum(-1).mean().item()


def val_batch(n, block_size):
    val = BinDataset("val")
    x = torch.stack([val.tokens(i * block_size, block_size) for i in range(n)])
    val.close()
    return x.to(DEV)

## Probe 1 — what does int8 actually cost?

Deterministic sweep over 2000 val windows (~1.02M tokens), so the deltas are exact
rather than sampled. **~27 s.**

Recorded result:

| variant | bpc | Δ bpc | KL nats | MB | shrink |
|---|---|---|---|---|---|
| fp32 baseline | 0.4985 | — | — | 104.0 | 1.00x |
| per-channel, all | 0.4986 | +0.0001 | 1.78e-04 | 26.2 | 3.97x |
| per-tensor, all | 0.4990 | +0.0005 | 1.08e-03 | 26.0 | 3.99x |
| per-channel, skip embeds | 0.4986 | +0.0001 | 1.41e-04 | 33.0 | 3.16x |

Three things it settles. int8 weight-only is **close to free** — 4x smaller for
0.02% of the quality. **KL separates what bpc cannot**: per-tensor is 5x worse in
bpc's fourth decimal but a clean **6.1x** in KL, at a fraction of the compute.
And **quantize the embeddings** — skipping them costs 3.97x → 3.16x to save
0.4e-04 nats, which is not a trade worth making.

In [3]:
MAX_WINDOWS = 2000

base = load()
bs = base.block_size
x = val_batch(8, bs)
with torch.no_grad():
    ref = base(x).float().log_softmax(-1)

val = BinDataset("val")
base_loss = full_loss(base, val, 16, bs, DEV, MAX_WINDOWS)
base_mb = nbytes(base.state_dict()) / 2**20
base_bpc = bits_per_char(base_loss)

VARIANTS = {
    "per-channel, all": dict(dim=-1),
    "per-tensor,  all": dict(dim=None),
    "per-channel, skip embeds": dict(
        dim=-1, skip=("token_embedding_table", "position_embedding_table", "lm_head")
    ),
}

print(f"{'variant':<26} {'bpc':>8} {'d bpc':>9} {'KL nats':>10} {'MB':>7} {'shrink':>8}")
print(f"{'fp32 baseline':<26} {base_bpc:>8.4f} {'--':>9} {'--':>10} {base_mb:>7.1f} {'1.00x':>8}")

for name, kw in VARIANTS.items():
    m = quantize_model(load(), **kw)
    kl = kl_vs(ref, m, x)
    bpc = bits_per_char(full_loss(m, val, 16, bs, DEV, MAX_WINDOWS))
    mb = nbytes(m.state_dict()) / 2**20
    print(f"{name:<26} {bpc:>8.4f} {bpc - base_bpc:>+9.4f} {kl:>10.2e} {mb:>7.1f} {base_mb / mb:>7.2f}x")
    del m
    torch.cuda.empty_cache()

val.close()

variant                         bpc     d bpc    KL nats      MB   shrink
fp32 baseline                0.4985        --         --   104.0    1.00x
per-channel, all             0.4986   +0.0001   1.78e-04    26.2    3.97x
per-tensor,  all             0.4990   +0.0005   1.08e-03    26.0    3.99x
per-channel, skip embeds     0.4986   +0.0001   1.41e-04    33.0    3.16x


## Probe 2 — where per-channel earns its keep

`quantize.py` samples four weights; this is every 2D weight in the model, sorted by
how badly a shared scale would treat its quietest row. **< 1 s.**

Recorded result: **worst 12.3x, median 3.5x, best 1.8x.**

| weight | ratio | per-tensor gives |
|---|---|---|
| `position_embedding_table.weight` | 12.3x | 10 of 127 |
| `blocks.{0,1}.attn.proj.weight` | 8.1x | 16 of 127 |
| `blocks.1.attn.qkv.weight` | 6.9x | 18 of 127 |
| `token_embedding_table.weight` = `lm_head.weight` | 2.7x | 47 of 127 |
| `blocks.6.ffwd.down.weight` | 1.8x | 70 of 127 |

Three readings. The **position table is the worst in the model** by a wide margin —
early positions are seen in every window and late ones only in long ones, so their
magnitudes diverge; a single scale would leave its quietest row 10 of 127 levels.
**Attention beats FFN for spread**, and **early blocks beat late ones**, monotonically
enough to look structural rather than incidental. And the tied pair appears twice with
identical ratios, which is the sharing from `quantize_model` showing up in the data.

Real outlier *features* — the 100x+ magnitudes that make per-tensor int8 fail outright
— only appear past roughly 6.7B parameters. At 27M this is a preview, not the full
version.

In [4]:
sd = torch.load(CKPT, map_location="cpu", weights_only=False)["model"]
QMAX = 127

rows = []
for name, w in sd.items():
    if w.ndim != 2:
        continue
    rm = w.float().abs().amax(-1)
    rows.append((( rm.max() / rm.min()).item(), name, w.shape[0], (QMAX * rm.min() / rm.max()).item()))

rows.sort(reverse=True)
print(f"{'weight':<34} {'rows':>5} {'ratio':>7} {'per-tensor gives':>18}")
for ratio, name, nrows, levels in rows:
    print(f"{name:<34} {nrows:>5} {ratio:>6.1f}x {f'{levels:.0f} of {QMAX}':>18}")

print(f"\nworst {rows[0][0]:.1f}x   median {sorted(r[0] for r in rows)[len(rows) // 2]:.1f}x   best {rows[-1][0]:.1f}x")

weight                              rows   ratio   per-tensor gives
position_embedding_table.weight      512   12.3x          10 of 127
blocks.1.attn.proj.weight            512    8.1x          16 of 127
blocks.0.attn.proj.weight            512    8.1x          16 of 127
blocks.1.attn.qkv.weight            1536    6.9x          18 of 127
blocks.2.attn.qkv.weight            1536    6.5x          20 of 127
blocks.3.attn.qkv.weight            1536    6.0x          21 of 127
blocks.6.attn.qkv.weight            1536    5.9x          21 of 127
blocks.4.attn.qkv.weight            1536    5.7x          22 of 127
blocks.0.attn.qkv.weight            1536    5.4x          23 of 127
blocks.7.attn.qkv.weight            1536    5.1x          25 of 127
blocks.5.attn.qkv.weight            1536    4.8x          26 of 127
blocks.7.attn.proj.weight            512    4.4x          29 of 127
blocks.1.ffwd.down.weight            512    4.3x          30 of 127
blocks.3.attn.proj.weight            512    3.9x

## Probe 3 — which layers actually hurt?

Quantize one layer at a time, measure KL against the fp32 logits. 35 layers, **~8 s.**

Recorded result — the ends of the table:

| layer | KL nats | MB saved |
|---|---|---|
| `lm_head` | 2.84e-05 | **−2.02** |
| `blocks.2.attn.qkv` | 1.61e-05 | +2.24 |
| `blocks.3.attn.qkv` | 1.19e-05 | +2.24 |
| … | | |
| `blocks.0.attn.qkv` | 5.32e-07 | +2.24 |
| `position_embedding_table` | 3.74e-07 | +0.75 |
| `blocks.0.attn.proj` | 3.57e-07 | +0.75 |

**Probe 2's prediction was wrong, and that is the finding.** The position table has
the widest rows in the model (12.3x) and is the *second least* sensitive layer here.
Row spread and sensitivity are different questions: per-channel scaling **already
neutralizes** spread — that is what it is for — so a wide ratio predicts how much
per-channel beats per-tensor, not how much quantizing that layer costs. Use probe 2
to choose the scheme, probe 3 to choose the layers.

**Errors add.** The 35 individual KLs sum to 1.71e-04 against 1.78e-04 measured
all-at-once — 96%. For small perturbations $\mathrm{KL} \approx \frac12 \delta^\top F \delta$,
so additivity says the per-layer logit perturbations $\delta_i$ are close to
orthogonal, which is what independent rounding errors should be. It also means
per-layer sensitivity is a **budget**: pick layers by KL-per-MB and the total is
predictable rather than emergent.

**`lm_head` is worst on both axes** — the most sensitive layer *and* the only one
whose MB saved is negative, because quantizing it alone unshares the tie. The size
column does not add up across rows for exactly that reason: the tied pair is one
tensor, and only quantizing both together (which `quantize_model` does by default)
actually saves its 2.02 MB.

Depth is not monotonic: `attn.qkv` sensitivity peaks at blocks 2–3 and is ~30x lower
at block 0.

**The ranking is stable, so 8 sequences is enough.** The sweep is deterministic — `round()`
has no randomness and the batch is fixed indices, so re-running it is bit-identical
(ρ = 1.000, max abs diff 0.0). The only axis that varies is *which* sequences you score on,
and it barely matters: four disjoint 8-sequence batches give mean ρ = 0.997 and the same
five layers in the top 5, and even n=4 gives ρ = 0.984. Measured once, not re-checked here,
because there is nothing left to check.


In [5]:
base = load()
bs = base.block_size
x = val_batch(8, bs)
with torch.no_grad():
    ref = base(x).float().log_softmax(-1)
base_mb = nbytes(base.state_dict()) / 2**20
kl_all = kl_vs(ref, quantize_model(load()), x)  # the whole model, for comparison

paths = quantizable(base)
rows = []
for target in paths:
    m = quantize_model(load(), skip=tuple(p for p in paths if p != target))
    rows.append((kl_vs(ref, m, x), target, base_mb - nbytes(m.state_dict()) / 2**20))
    del m
    torch.cuda.empty_cache()

rows.sort(reverse=True)
print(f"{'layer':<30} {'KL nats':>10} {'MB saved':>10} {'KL per MB':>11}")
for kl, name, saved in rows:
    per_mb = f"{kl / saved:.2e}" if saved > 0 else "--"
    print(f"{name:<30} {kl:>10.2e} {saved:>+10.2f} {per_mb:>11}")

total = sum(r[0] for r in rows)
print(f"\nsum of individual KLs {total:.2e}   all-at-once {kl_all:.2e}   ratio {total / kl_all:.2f}")

layer                             KL nats   MB saved   KL per MB
lm_head                          2.84e-05      -2.02          --
blocks.2.attn.qkv                1.61e-05      +2.24    7.16e-06
blocks.3.attn.qkv                1.19e-05      +2.24    5.28e-06
blocks.4.attn.qkv                8.17e-06      +2.24    3.64e-06
blocks.1.attn.qkv                7.05e-06      +2.24    3.14e-06
blocks.0.ffwd.down               6.65e-06      +1.97    3.38e-06
blocks.7.ffwd.gate_up            6.16e-06      +3.93    1.57e-06
blocks.6.ffwd.gate_up            6.03e-06      +3.93    1.54e-06
blocks.5.ffwd.gate_up            5.69e-06      +3.93    1.45e-06
blocks.3.ffwd.down               5.48e-06      +1.97    2.79e-06
blocks.4.ffwd.gate_up            5.45e-06      +3.93    1.39e-06
blocks.3.ffwd.gate_up            4.92e-06      +3.93    1.25e-06
blocks.7.ffwd.down               4.70e-06      +1.97    2.39e-06
token_embedding_table            4.40e-06      -2.02          --
blocks.6.ffwd.down       

## Probe 4 — is KL-per-MB a valid budget?

Probe 3 found the per-layer errors are ~96% additive, which is what would license picking
layers greedily. This tests that directly: sort by KL/MB, quantize the cheapest $N$, and
compare the **measured** total KL against the **predicted** sum of the individual ones.
34 steps, **~16 s.**

Recorded result — measured/predicted across the whole sweep:

| N units | MB saved | predicted KL | measured KL | meas/pred |
|---|---|---|---|---|
| 1 | 2.2 | 5.32e-07 | 5.32e-07 | 1.00x |
| 8 | 19.7 | 1.26e-05 | 1.23e-05 | 0.98x |
| 16 | 43.1 | 4.54e-05 | 4.45e-05 | 0.98x |
| 24 | 57.9 | 7.61e-05 | 7.39e-05 | 0.97x |
| 32 | 69.6 | 1.22e-04 | 1.24e-04 | 1.01x |
| 34 (all) | 77.8 | 1.71e-04 | 1.78e-04 | 1.04x |

**Never worse than 4% off, at any point on the curve.** So KL/MB is a real budget, not a
heuristic: you can pick a KL ceiling, take layers until you hit it, and the total lands
where the sum said it would.

The trade it exposes is the case for mixed precision. Protecting just the **2** priciest
units gives up 8.2 MB of the 77.8 MB available (11%) and removes **30%** of the error;
protecting **6** gives up 18% of the savings to remove **47%**. That is the shape int4
should exploit — cheap layers to 4 bits, the handful of expensive ones left at 8.

Tied weights are grouped into one **unit** (35 layers → 34), because the tied pair is one
tensor and therefore one decision — quantizing half of it unshares and costs memory.

In [6]:
base = load()
x = val_batch(8, base.block_size)
with torch.no_grad():
    ref = base(x).float().log_softmax(-1)
base_mb = nbytes(base.state_dict()) / 2**20
paths = quantizable(base)


def get(m, path):
    for part in path.split("."):
        m = getattr(m, part)
    return m


# a tied pair is one tensor, so it is one decision -- group by storage
groups = {}
for path in paths:
    groups.setdefault(get(base, path).weight.data_ptr(), []).append(path)
units = list(groups.values())
print(f"{len(paths)} layers -> {len(units)} units; tied: {[u for u in units if len(u) > 1]}")


def quantize_only(chosen):
    m = quantize_model(load(), skip=tuple(p for p in paths if p not in chosen))
    return kl_vs(ref, m, x), base_mb - nbytes(m.state_dict()) / 2**20, m


rows = []
for unit in units:
    kl, mb, m = quantize_only(unit)
    rows.append((kl, mb, unit))
    del m
    torch.cuda.empty_cache()

rows.sort(key=lambda r: r[0] / r[1])  # cheapest error per MB first

print(f"\n{'N':>3} {'MB saved':>9} {'predicted':>11} {'measured':>11} {'meas/pred':>10}")
for n in range(1, len(rows) + 1):
    chosen = [p for r in rows[:n] for p in r[2]]
    meas, mb, m = quantize_only(chosen)
    pred = sum(r[0] for r in rows[:n])
    del m
    torch.cuda.empty_cache()
    if n in (1, len(rows)) or n % 4 == 0:
        print(f"{n:>3} {mb:>9.1f} {pred:>11.2e} {meas:>11.2e} {meas / pred:>9.2f}x")

print("\ncheapest 5:", [r[2][0] for r in rows[:5]])
print("priciest 5:", [r[2][0] for r in rows[-5:]])

35 layers -> 34 units; tied: [['token_embedding_table', 'lm_head']]

  N  MB saved   predicted    measured  meas/pred
  1       2.2    5.32e-07    5.32e-07      1.00x
  4       7.7    2.44e-06    2.54e-06      1.04x
  8      19.7    1.26e-05    1.23e-05      0.98x
 12      27.4    2.21e-05    2.17e-05      0.98x
 16      43.1    4.54e-05    4.45e-05      0.98x
 20      50.0    5.78e-05    5.62e-05      0.97x
 24      57.9    7.61e-05    7.39e-05      0.97x
 28      63.6    9.50e-05    9.35e-05      0.98x
 32      69.6    1.22e-04    1.24e-04      1.01x
 34      77.8    1.71e-04    1.78e-04      1.04x

cheapest 5: ['blocks.0.attn.qkv', 'blocks.1.ffwd.gate_up', 'blocks.0.attn.proj', 'position_embedding_table', 'blocks.1.ffwd.down']
priciest 5: ['blocks.4.attn.proj', 'blocks.3.attn.proj', 'blocks.3.attn.qkv', 'token_embedding_table', 'blocks.2.attn.qkv']


## Probe 5 — how narrow can the weights go?

`pack_bits` handles any width, so this is the whole curve rather than 4-vs-8.
Sizes are measured, not projected. **~40 s.**

Recorded result:

| | MB | shrink | KL | bpc | Δ bpc | KL vs previous |
|---|---|---|---|---|---|---|
| int2 | 6.7 | 15.47x | 7.61e+00 | 3.0431 | +2.5445 | — |
| int3 | 10.0 | 10.43x | 5.15e-01 | 0.6979 | +0.1994 | 14.8x |
| int4 | 13.2 | 7.87x | 6.10e-02 | 0.5229 | +0.0244 | 8.4x |
| int5 | 16.5 | 6.31x | 1.25e-02 | 0.5035 | +0.0050 | 4.9x |
| int6 | 19.7 | 5.27x | 2.93e-03 | 0.4997 | +0.0012 | 4.3x |
| int8 | 26.2 | 3.97x | 1.78e-04 | 0.4986 | +0.0001 | 16.5x |
| **fp32** | **104.0** | **1.00x** | — | **0.4985** | — | — |

**int6 is the sweet spot, and it is not one of the widths anybody names.** 5.27x for
+0.0012 bpc — a 0.24% quality cost against int4's 4.9%.

**One bit is worth 4x, and the law holds from 4 bits up.** 4→5 measures 4.9x, 5→6
measures 4.3x, and 6→8 measures **16.5x** against 16x predicted. Below 4 bits it breaks
badly (8.4x for 3→4, 14.8x for 2→3): $\mathrm{KL} \approx \frac12 \delta^\top F \delta$
is a *small*-perturbation result, and at 2–3 bits the perturbation is not small. int2
lands at bpc 3.04 — **worse than the untrained model's 2.96**, i.e. destroyed rather
than degraded.

In [7]:
val = BinDataset("val")
base = load()
bs = base.block_size
x = val_batch(8, bs)
with torch.no_grad():
    ref = base(x).float().log_softmax(-1)
base_mb = nbytes(base.state_dict()) / 2**20
base_bpc = bits_per_char(full_loss(base, val, 16, bs, DEV, 2000))
print(f"fp32  {base_mb:.1f} MB  bpc {base_bpc:.4f}\n")

print(f"{'':>6} {'MB':>7} {'shrink':>7} {'KL':>10} {'bpc':>8} {'d bpc':>8} {'KL ratio':>9}")
prev = None
for b in (2, 3, 4, 5, 6, 8):
    m = quantize_model(load(), bits=b)
    kl = kl_vs(ref, m, x)
    mb = nbytes(m.state_dict()) / 2**20
    bpc = bits_per_char(full_loss(m, val, 16, bs, DEV, 2000))
    ratio = f"{prev / kl:.1f}x" if prev else "--"
    print(f"int{b:<3} {mb:>7.1f} {base_mb / mb:>6.2f}x {kl:>10.2e} {bpc:>8.4f} "
          f"{bpc - base_bpc:>+8.4f} {ratio:>9}")
    prev = kl
    del m
    torch.cuda.empty_cache()
val.close()

fp32  104.0 MB  bpc 0.4985

            MB  shrink         KL      bpc    d bpc  KL ratio
int2       6.7  15.47x   7.61e+00   3.0431  +2.5445        --
int3      10.0  10.43x   5.15e-01   0.6979  +0.1994     14.8x
int4      13.2   7.87x   6.10e-02   0.5229  +0.0244      8.4x
int5      16.5   6.31x   1.25e-02   0.5035  +0.0050      4.9x
int6      19.7   5.27x   2.93e-03   0.4997  +0.0012      4.3x
int8      26.2   3.97x   1.78e-04   0.4986  +0.0001     16.5x


## Probe 6 — mixed precision, allocated properly

Probe 4 showed the per-layer errors add, and probe 5 showed one bit is worth 4x. Together
they give a closed-form allocation. Minimising $\sum_i A_i 4^{-b_i}$ subject to a bit
budget $\sum_i n_i b_i = B$:

$$b_i \;=\; \log_4\!\left(\frac{A_i}{n_i}\right) + c$$

Bits scale with the log of **per-weight** sensitivity. The normalisation by $n_i$ matters —
a large layer has high total KL just for being large, and ranking on raw KL protects it for
the wrong reason. **~60 s.**

Recorded result — **per-weight sensitivity spans 28x, so the optimal spread is
$\log_4 28 = 2.4$ bits** (1.4 bits from p10 to p90):

| | MB | Δ bpc |
|---|---|---|
| uniform int5 | 16.5 | +0.0050 |
| mixed `[(4,7), (5,20), (6,8)]` | 16.4 | **+0.0041** |
| uniform int6 | 19.7 | +0.0012 |
| mixed `[(4,2), (5,6), (6,22), (7,5)]` | 19.1 | +0.0012 |

**~18% less error at the same size.** Real, and bounded: the win is exactly the distance
from uniform to the equilibrium where every $\mathrm{KL}_i/n_i$ is equal, which is
$\log_4$ of the spread.

**Why moving a bit pays.** Each layer's error is multiplicative in its own width: take a
bit from layer *j* and its KL goes ×4 (costing $3\,\mathrm{KL}_j$); give it to layer *i*
and its KL goes ÷4 (saving $0.75\,\mathrm{KL}_i$). Profitable whenever
$\mathrm{KL}_i > 4\,\mathrm{KL}_j$, and a 28x spread clears that easily. Bits are **moved**,
not removed — the totals match to 0.3%, so the small size differences above are rounding,
not a saving.

**The tempting explanation is wrong.** "Downgrade the big insensitive layers, upgrade the
small sensitive ones" would let you win on both axes — but `correlation(size, KL per
weight) = -0.05` here, and the mean layer size in each bit bucket is within 13%. Size and
sensitivity are unrelated in this model; the win is purely the swap argument above.

An earlier version of this probe tested **int8/int4** and concluded mixed precision loses to
uniform. That was a 4-bit gap against a 2.4-bit optimal spread — it overshoots on both ends,
starving the cheap layers to give the expensive ones bits they cannot use.

In [8]:
val = BinDataset("val")
base = load()
bs = base.block_size
x = val_batch(8, bs)
with torch.no_grad():
    ref = base(x).float().log_softmax(-1)
base_mb = nbytes(base.state_dict()) / 2**20
base_bpc = bits_per_char(full_loss(base, val, 16, bs, DEV, 2000))
INT8_FLOOR = 1.78e-04  # what stays even at 8 bits; subtract it to isolate the width


def get(m, path):
    for part in path.split("."):
        m = getattr(m, part)
    return m


paths = quantizable(base)
groups = {}
for path in paths:
    groups.setdefault(get(base, path).weight.data_ptr(), []).append(path)
units = [(u, get(base, u[0]).weight.numel()) for u in groups.values()]

# A_i / n_i, measured by putting one unit at a reference width and the rest at 8
rows = []
for u, n in units:
    m = quantize_model(load(), bits=8, override={p: 6 for p in u})
    kl = max(kl_vs(ref, m, x) - INT8_FLOOR, 1e-12)
    rows.append((kl, n, u))
    del m
    torch.cuda.empty_cache()

per_w = sorted(kl / n for kl, n, _ in rows)
spread = per_w[-1] / per_w[0]
print(f"per-weight sensitivity spread {spread:.0f}x -> optimal bit spread "
      f"log4({spread:.0f}) = {math.log(spread, 4):.2f} bits")

# and the intuitive story -- big layers are the insensitive ones -- is not true here
ns = [n for _, n, _ in rows]
ss = [kl / n for kl, n, _ in rows]
mn, ms = sum(ns) / len(ns), sum(ss) / len(ss)
cov = sum((n - mn) * (s - ms) for n, s in zip(ns, ss))
den = math.sqrt(sum((n - mn) ** 2 for n in ns) * sum((s - ms) ** 2 for s in ss))
print(f"correlation(size, KL per weight) = {cov / den:+.3f}\n")


def allocate(offset):
    """b_i = log4(A_i / n_i) + offset, clamped to widths we can pack."""
    return {p: max(3, min(8, round(math.log(kl / n, 4) + offset)))
            for kl, n, u in rows for p in u}


def report(label, model):
    mb = nbytes(model.state_dict()) / 2**20
    bpc = bits_per_char(full_loss(model, val, 16, bs, DEV, 2000))
    print(f"{label:<38} {mb:>7.1f} {kl_vs(ref, model, x):>10.2e} {bpc - base_bpc:>+9.4f}")


print(f"{'scheme':<38} {'MB':>7} {'KL':>10} {'d bpc':>9}")
for b in (5, 6):
    m = quantize_model(load(), bits=b)
    report(f"uniform int{b}", m)
    del m
    torch.cuda.empty_cache()

for off in (21.8, 22.6):
    alloc = allocate(off)
    hist = {}
    for v in alloc.values():
        hist[v] = hist.get(v, 0) + 1
    m = quantize_model(load(), bits=8, override=alloc)
    report(f"mixed {sorted(hist.items())}", m)
    del m
    torch.cuda.empty_cache()
val.close()

per-weight sensitivity spread 28x -> optimal bit spread log4(28) = 2.39 bits
correlation(size, KL per weight) = -0.051

scheme                                      MB         KL     d bpc
uniform int5                              16.5   1.25e-02   +0.0050
uniform int6                              19.7   2.93e-03   +0.0012
mixed [(4, 7), (5, 20), (6, 8)]           16.4   1.01e-02   +0.0041
mixed [(4, 2), (5, 6), (6, 22), (7, 5)]    19.1   3.24e-03   +0.0012


## Probe 7 — is any of it faster?

Everything so far measured size and quality. This measures the clock, and the memory a
forward pass *adds* on top of the resident weights. **~60 s.**

Recorded result:

| | resident MB | +MB per forward | | | slowdown vs fp32 | | |
|---|---|---|---|---|---|---|---|
| | | B=1,T=1 | B=1,T=512 | B=16,T=512 | B=1,T=1 | B=1,T=512 | B=16,T=512 |
| fp32 | 104.0 | 0 | 15 | 232 | 1x | 1x | 1x |
| int8 | 26.2 | 16 | 17 | 232 | ~1.3x | ~1.1x | ~1.0x |
| int6 | 19.7 | 34 | 35 | 232 | ~3.8x | ~1.6x | ~1.1x |
| int4 | 13.2 | 18 | 19 | 232 | ~3.6x | ~1.4x | ~1.0x |

The `+MB` columns are exact and reproduce run to run. Latency is a rounded ratio on
purpose — absolute ms jitters a few percent every execution, so two-decimal timings could
only ever disagree with the cell below.

**Nothing is faster, and the shape of the loss is inverted from a real kernel.**
Dequantization cost depends on weight size, so it is paid *per forward*, while matmul cost
scales with tokens. Hence ~3.6x at decode and ~1x at batch. A real int8 kernel does the
opposite — it helps **decode**, which is memory-bandwidth-bound and reads 4x fewer bytes,
and helps least at prefill, which is compute-bound. This implementation is slowest exactly
where quantization is supposed to win.

**But memory does pay, on both axes.** Total in flight is resident + transient:

| | resident | transient | **total** |
|---|---|---|---|
| fp32 | 104.0 | 0 | **104.0** |
| int8 | 26.2 | 16 | **42.2** |
| int6 | 19.7 | 34 | **53.7** |
| int4 | 13.2 | 18 | **31.2** |

int4 is smallest both at rest and in use — 3.3x less memory than fp32 during a decode step,
for ~3.6x the latency. That is the real trade this file buys.

**int6 costs more in flight than int8**, despite being smaller on disk, and the reason is
`acc_dtype`: a 6-bit block is 24 bits wide so it needs an `int32` container, while a 4-bit
block is 8 bits and fits `int16`. Twice the container, twice the temporaries. Rebuilding
one 8 MB weight costs 16 MB at int8, 18 at int4, 34 at int6.

**The bug this probe caught.** The first version of this table read 60 MB transient for
both int4 and int6, and concluded that sub-byte packing gives its memory saving back.
That was wrong. `sum()` over an integer tensor **promotes to int64** unless given
`dtype=`, so `acc_dtype` applied to the first step and was undone at the second — every
temporary downstream was int64 regardless of width. Passing `dtype=dt` through the sum
took int4 from 60 MB to 18 and from ~5.5x to ~3.6x. A conclusion about a *technique* was
really a fact about one missing keyword argument.

In [3]:
import time

SHAPES = [(1, 1), (1, 512), (16, 512)]  # decode, prefill, batched prefill


def bench(model, B, T, iters=20):
    """ms per forward, and the peak memory the forward itself adds.

    A delta, not max_memory_allocated: that counts everything resident, so the
    absolute number would depend on what else this notebook is holding.
    """
    x = torch.randint(0, 4096, (B, T), device=DEV)
    torch.cuda.empty_cache()  # same allocator state for every variant
    with torch.no_grad():
        for _ in range(3):
            model(x)  # warm up the allocator and any autotuning
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        before = torch.cuda.memory_allocated()
        t0 = time.perf_counter()
        for _ in range(iters):
            model(x)
        torch.cuda.synchronize()
        ms = (time.perf_counter() - t0) * 1000 / iters
    return ms, (torch.cuda.max_memory_allocated() - before) / 2**20


print(f"{'':>6} {'resident':>9}   " + "   ".join(f"{f'B={b},T={t}':>16}" for b, t in SHAPES))
print(f"{'':>6} {'MB':>9}   " + "   ".join(f"{'ms':>7} {'+MB':>8}" for _ in SHAPES))
for name, bits in (("fp32", None), ("int8", 8), ("int6", 6), ("int4", 4)):
    m = load()
    if bits:
        quantize_model(m, bits=bits)
    cells = [f"{ms:>7.2f} {mb:>8.0f}" for ms, mb in (bench(m, B, T) for B, T in SHAPES)]
    print(f"{name:>6} {nbytes(m.state_dict()) / 2**20:>9.1f}   " + "   ".join(cells))
    del m
    torch.cuda.empty_cache()

# and where the sub-byte peak goes: rebuilding ONE weight
w = torch.randn(4096, 512, device=DEV)
print(f"\nrebuilding one {w.numel() * 4 / 2**20:.0f} MB weight:")
for bits in (8, 6, 4):
    q, sc = quantize_packed(w, -1, bits)
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    before = torch.cuda.memory_allocated()
    dequantize(q if bits == 8 else unpack_bits(q, bits), sc)
    torch.cuda.synchronize()
    stored = (q.numel() * q.element_size() + sc.numel() * 4) / 2**20
    transient = (torch.cuda.max_memory_allocated() - before) / 2**20
    print(f"  int{bits}: stored {stored:>5.2f} MB   transient {transient:>6.2f} MB")
del w
torch.cuda.empty_cache()


        resident            B=1,T=1          B=1,T=512         B=16,T=512
              MB        ms      +MB        ms      +MB        ms      +MB
  fp32     104.0      1.14        0      5.84       15     89.33      232
  int8      26.2      1.54       16      6.55       17     90.78      232
  int6      19.7      4.08       34      9.11       35     94.80      232
  int4      13.2      4.02       18      8.16       19     92.36      212

rebuilding one 8 MB weight:
  int8: stored  2.02 MB   transient  16.00 MB
  int6: stored  1.52 MB   transient  34.00 MB
  int4: stored  1.02 MB   transient  18.00 MB
